# W4 Lab — The Agent Loop, Built by Hand

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ralbu85/stml_2026/blob/main/lectures/week04/W4_lab_loop.ipynb)

**Goal.** Build the Call → Judge → Execute → Reinject loop of notes Ch. 4 from
nothing — about twenty-five lines, the Judge step your fill-in — and measure on a
five-question evalset, re-scored in this week's homework, that ReAct's written
Thought beats Act-only on multi-hop questions.

Why this week: W3's multi-step requests worked because `aisuite` ran the loop out of
sight, and a borrowed loop cannot be shaped or read — this lab makes the protocol,
the bound, and the record yours (notes Ch. 4 §4.1).

The path: setup → the hidden loop replayed → the loop by hand (plain-function tools,
the ReAct protocol, the Judge fill-in ✍️) → Act-only vs ReAct, measured ✍️ (core) →
trace reading → the step bound → completion.

> Built for this course. The from-scratch construction follows the pattern of
> Hugging Face's *Agents Course*, unit 1 (the dummy-agent notebook); the protocol
> and traces follow ReAct (Yao et al., 2022).

*Runtime:* Google Colab, top-to-bottom, ~80 minutes. Cells marked ✍️ ask for a
fill-in or a written prediction.

## 1. Setup

### 1.1 Installation
*Do:* run the install cell below.


In [ ]:
%pip install -q "aisuite[openai,anthropic]"


### 1.2 API key and model

Paste your key between the quotes, exactly as in the previous labs.


In [ ]:
import os

os.environ["OPENAI_API_KEY"] = "PASTE-YOUR-KEY-HERE"

MODEL = "openai:gpt-4o-mini"          # alt: "anthropic:claude-haiku-4-5"


### 1.3 Client and helpers
*Do:* run the cell unchanged.


In [ ]:
import aisuite

client = aisuite.Client()


def chat(messages, **kwargs):
    """Messages -> assistant text (single call, no tools)."""
    response = client.chat.completions.create(model=MODEL, messages=messages, **kwargs)
    return response.choices[0].message.content


def ask(prompt, system=None, **kwargs):
    """One-shot convenience wrapper around chat()."""
    messages = ([{"role": "system", "content": system}] if system else []) + [
        {"role": "user", "content": prompt}]
    return chat(messages, **kwargs)


### 1.4 Verification

*Do:* run the cell; it must print `ready`.


In [ ]:
print(ask("Reply with exactly: ready"))


## 2. The Hidden Loop, Replayed

The cell below replays W3's multi-step shape on two small tools; everything on the
model side happens inside `create(...)`.

*Do:* run the cell, then answer from the output alone: how many model calls
happened? You cannot tell — the loop ran where you cannot see it.

In [ ]:
PAPER_CATALOG = {
    "react":            {"title": "ReAct: Synergizing Reasoning and Acting in Language Models",
                         "authors": "Yao et al.", "year": 2022},
    "chain-of-thought": {"title": "Chain-of-Thought Prompting Elicits Reasoning in LLMs",
                         "authors": "Wei et al.", "year": 2022},
    "toolformer":       {"title": "Toolformer: LMs Can Teach Themselves to Use Tools",
                         "authors": "Schick et al.", "year": 2023},
    "self-consistency": {"title": "Self-Consistency Improves Chain of Thought Reasoning",
                         "authors": "Wang et al.", "year": 2022},
}


def calculator(expression: str) -> str:
    """Evaluates an arithmetic expression and returns the result as a string.

    Args:
        expression: Python arithmetic syntax, digits and + - * / ( ) only.
    """
    allowed = set("0123456789+-*/(). ")
    if not expression or not set(expression) <= allowed:
        return f"(calculator error: unsupported characters in {expression!r})"
    try:
        return str(eval(expression, {"__builtins__": {}}, {}))
    except Exception as exc:
        return f"(calculator error: {exc})"


def paper_lookup(topic: str) -> str:
    """Looks up a paper in the course catalog; returns title, authors, year.

    Args:
        topic: one of: react, chain-of-thought, toolformer, self-consistency.
    """
    entry = PAPER_CATALOG.get(topic.strip().lower())
    if entry is None:
        return (f"(unknown topic {topic!r}. Known topics: "
                + ", ".join(sorted(PAPER_CATALOG)) + ")")
    return f"{entry['title']} — {entry['authors']}, {entry['year']}"


response = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user",
               "content": "Which was published earlier, Toolformer or chain-of-thought, "
                          "according to the catalog?"}],
    tools=[calculator, paper_lookup],
    max_turns=5,
)
print(response.choices[0].message.content)


To set the protocol, enforce the bound, and read the record, the loop has to be ours
(notes Ch. 4 §4.1) — the rest of this lab builds it.

## 3. The Loop by Hand

### 3.1 Tools as plain functions

No `tools=` parameter from here on. The registry is a dict from tool name to
function; execution failures come back as result strings, never exceptions —
an error the model can read is an error it can correct (notes Ch. 3 §3.5).
*Do:* run the cell unchanged.


In [ ]:
TOOLS = {
    "calculator": calculator,
    "paper_lookup": paper_lookup,
}


def run_tool(name: str, tool_input: str) -> str:
    """Executes a registered tool; every failure becomes a result string."""
    if name not in TOOLS:
        return f"(unregistered tool {name!r}. Available: " + ", ".join(sorted(TOOLS)) + ")"
    try:
        return TOOLS[name](tool_input)
    except Exception as exc:
        return f"(tool {name!r} raised: {exc})"


### 3.2 The protocol — ReAct

The system prompt below is the specification of notes Ch. 4 §4.3, verbatim: every
turn is a Thought + Action or a Thought + Final Answer, and the Observation is never
written by the model — the loop attaches it.
*Do:* run the cell, then read the protocol text once, end to end.

In [ ]:
REACT_SYSTEM = """You are an agent that thinks and acts step by step. On every turn, answer in exactly one of the two formats below.

When a tool is needed:
Thought: <your reasoning so far>
Action: {"tool": "<tool name>", "input": "<input>"}

Available tools:
- calculator: evaluates an arithmetic expression (Python syntax, e.g. "2023 - 2022").
- paper_lookup: looks up a paper by topic key. Keys: react, chain-of-thought, toolformer, self-consistency.

When finalizing the answer:
Thought: <final reasoning>
Final Answer: <answer>

Observation: is filled in by the system — never write it yourself. Write nothing after Action."""


### 3.3 The Judge ✍️

The Judge step parses one model output into the loop's three branches (notes
Ch. 4 §4.1). This is the fill-in — the loop below runs as-is, but with the
starter judge it can only reinject errors.

Write `judge(text)` so that it returns exactly one of:

- `("final", answer)` — the text contains a `Final Answer:` line; `answer` is what
  follows it (strip whitespace);
- `("action", name, tool_input)` — the text contains an `Action:` line whose JSON
  parses to a dict with string fields `"tool"` and `"input"`;
- `("error", message)` — anything else; `message` must say what is wrong so the
  model can correct it.

Hints — the ingredients, in order:
1. `re.search(r"Final Answer:\s*(.*)", text, re.S)` — check this **first**: a
   termination turn must win even if the model also rambles.
2. `re.search(r"Action:\s*(\{.*?\})", text, re.S)` then `json.loads(...)` inside
   `try/except` — malformed JSON is an `("error", ...)`, not a crash.
3. Validate the parsed dict: is `"tool"` present? is the name in `TOOLS`? (An
   unknown name may also be left to `run_tool`, which already answers it readably.)

In [ ]:
import json
import re


### FILL IN (START) ###
def judge(text: str):
    """Model output -> ("final", answer) | ("action", name, input) | ("error", msg)."""
    return ("error", "judge not implemented yet — parse Final Answer and Action here")
### FILL IN (END) ###


# Quick self-test: three vectors the finished judge must handle.
print(judge("Thought: done.\nFinal Answer: 42"))
print(judge('Thought: look it up.\nAction: {"tool": "paper_lookup", "input": "react"}'))
print(judge("I just feel like chatting."))


### 3.4 The loop

Twenty-five lines: Call increments and checks the bound, Judge branches, Execute
runs the tool, Reinject appends and returns to Call. The exits are the Final Answer
and the bound — and the bound reports itself as a failure instead of inventing an
answer (notes Ch. 4 §4.6).

*Do:* before running, match each block against the four steps of Figure 4.1 in the
notes.

In [ ]:
def agent_loop(question, system=REACT_SYSTEM, max_steps=6, verbose=True):
    """Runs the hand-built agent loop; returns (answer, trace)."""
    messages = [{"role": "system", "content": system},
                {"role": "user", "content": question}]
    trace = []
    for step in range(1, max_steps + 1):                       # Call
        output = chat(messages)
        trace.append(("model", output))
        if verbose:
            print(f"--- step {step} ---\n{output}")
        verdict = judge(output)                                # Judge
        if verdict[0] == "final":
            return verdict[1], trace
        if verdict[0] == "action":
            observation = run_tool(verdict[1], verdict[2])     # Execute
        else:
            observation = verdict[1]                           # format error, reinjected
        trace.append(("observation", observation))
        if verbose:
            print(f"Observation: {observation}")
        messages.append({"role": "assistant", "content": output})   # Reinject
        messages.append({"role": "user", "content": f"Observation: {observation}"})
    failure = f"(no Final Answer within max_steps={max_steps}; see trace)"
    return failure, trace


### 3.5 First run — a multi-hop question

The question below needs two lookups and one subtraction; no single tool result
contains the answer.

*Do:* run the cell and read the trace: where was the ground for each action written
before the action was taken?

In [ ]:
MULTI_HOP = ("How many years passed between chain-of-thought and Toolformer, "
             "according to the catalog?")

answer, trace = agent_loop(MULTI_HOP)
print("\nANSWER:", answer)


## 4. Act-Only versus ReAct, Measured ✍️ (core)

### 4.1 The Act-only protocol

Same loop, same tools, one change: the protocol no longer asks for a Thought — the
ReAct paper's contrast condition (notes Ch. 4 §4.2).
*Do:* run the cell unchanged.

In [ ]:
ACT_ONLY_SYSTEM = """You are an agent that acts step by step. On every turn, answer in exactly one of the two formats below. Write nothing else.

When a tool is needed:
Action: {"tool": "<tool name>", "input": "<input>"}

Available tools:
- calculator: evaluates an arithmetic expression (Python syntax, e.g. "2023 - 2022").
- paper_lookup: looks up a paper by topic key. Keys: react, chain-of-thought, toolformer, self-consistency.

When finalizing the answer:
Final Answer: <answer>

Observation: is filled in by the system — never write it yourself. Write nothing after Action."""

answer_act, trace_act = agent_loop(MULTI_HOP, system=ACT_ONLY_SYSTEM)
print("\nANSWER (act-only):", answer_act)


### 4.2 The mini evalset

Five questions with checkable answers over the same tools — one single-hop, the rest
multi-hop. This is the course's standing instrument: the homework re-scores it after
adding the repetition guard, and later weeks return to it
(`labs/data/mini_evalset.jsonl` carries the canonical copy). The scorer is a keyword
check on the final answer — code-graded, in next week's sense.

In [ ]:
MINI_EVALSET = [
    {"question": "In which year was ReAct published, according to the catalog?",
     "keywords": ["2022"]},
    {"question": "Who are the authors of Toolformer, according to the catalog?",
     "keywords": ["schick"]},
    {"question": "Which was published earlier according to the catalog, Toolformer or chain-of-thought?",
     "keywords": ["chain-of-thought"]},
    {"question": "How many years passed between chain-of-thought and Toolformer, according to the catalog?",
     "keywords": ["1"]},
    {"question": "Who is the first author of self-consistency, and in which year was it published?",
     "keywords": ["wang", "2022"]},
]


def accuracy(system_prompt, evalset, verbose=False):
    """Fraction of items whose final answer contains every expected keyword."""
    correct = 0
    for item in evalset:
        answer, _ = agent_loop(item["question"], system=system_prompt, verbose=False)
        hit = all(k.lower() in str(answer).lower() for k in item["keywords"])
        correct += hit
        if verbose:
            print(f"{'PASS' if hit else 'FAIL':4}  {item['question'][:60]}  ->  {str(answer)[:60]}")
    return correct


### 4.3 Run ✍️

Target: with your finished judge, **ReAct ≥ 4/5**, and at least as high as Act-only.

*Do:* prediction first, written down: on which of the five items could Act-only
plausibly tie ReAct, and why? (Think: which item needs no intermediate judgment?)
Then run and compare.


In [ ]:
react_score = accuracy(REACT_SYSTEM, MINI_EVALSET, verbose=True)
print(f"\nReAct:    {react_score}/5")
act_score = accuracy(ACT_ONLY_SYSTEM, MINI_EVALSET, verbose=True)
print(f"Act-only: {act_score}/5")

TARGET_REACT = 4


Single-hop items forgive a missing Thought — the cue for the only action sits in the
question itself; the gap opens on multi-hop items, where the termination judgment
must integrate facts no single observation states (notes Ch. 4 §4.2).

## 5. Trace Reading — a Failing Run

The question below asks for a paper the catalog does not hold. The lookup answers
readably ("unknown topic"); some runs then admit ignorance, some fabricate a year.

*Do:* run the cell, then apply the procedure of notes Ch. 4 §4.6 to the trace: find
the **first** point that conflicts with fact or task, classify it (Thought / Action /
Observation), and name the fix before reading on.

In [ ]:
answer_fail, trace_fail = agent_loop(
    "In which year was Reflexion published, according to the catalog?")
print("\nANSWER:", answer_fail)


If the run fabricated a year, the first error is in a Thought — the judgment to
answer from memory — so the fix belongs in the prompt: the Judge, the tools, and the
observations all behaved. One added line closes it:

In [ ]:
GUARDED_SYSTEM = REACT_SYSTEM + (
    "\nIf a tool result says a topic is unknown, do not guess from memory - "
    "give a Final Answer that says the catalog does not contain it.")

answer_guarded, _ = agent_loop(
    "In which year was Reflexion published, according to the catalog?",
    system=GUARDED_SYSTEM)
print("\nANSWER (guarded):", answer_guarded)


### Exercise — the bound as an honest exit ✍️

Prediction, written down first: rerun the 3.5 multi-hop question with `max_steps=2` —
what exactly does the returned answer say, and why is that better than the loop
inventing its best guess at the cutoff?

*Do:* run the cell, compare with your prediction, then check your reasoning against
notes Ch. 4 §4.6 (the bound report).

In [ ]:
answer_cut, trace_cut = agent_loop(MULTI_HOP, max_steps=2)
print("\nANSWER:", answer_cut)


## 6. Completion Check

All rows must read `PASS` before submission; grading checks these structural facts,
never prose quality.


In [ ]:
_j1 = judge("Thought: done.\nFinal Answer: 42")
_j2 = judge('Thought: x.\nAction: {"tool": "calculator", "input": "1+1"}')
_j3 = judge("I just feel like chatting.")

completion = {
    "judge parses a Final Answer turn":
        _j1[0] == "final" and str(_j1[1]).strip() == "42",
    "judge parses an Action turn":
        _j2[0] == "action" and _j2[1] == "calculator" and _j2[2] == "1+1",
    "judge reports a format error readably":
        _j3[0] == "error" and len(str(_j3[1])) > 10,
    f"ReAct accuracy >= {TARGET_REACT}/5":
        react_score >= TARGET_REACT,
    "ReAct >= Act-only on the evalset":
        react_score >= act_score,
    "bounded run reports its failure":
        "max_steps" in str(answer_cut),
}

for item, ok in completion.items():
    print(f"{'PASS' if ok else 'FAIL':4}  {item}")
print("\nLAB COMPLETE" if all(completion.values()) else "\nNOT COMPLETE YET")


---

Next week reads what this loop refuses to read: the Final Answer it just committed.
Reflection feeds it back; evaluation makes "it got better" a number (notes Ch. 5).
Reference answers for this lab and the homework:
`labs/checkpoints/week04/solution.py`, published after the homework deadline.
